# atmoz 0.1.0 — hands-on test notebook

Work top to bottom. Each section says what *should* happen. Sections 3-4 need an AirNow API key
(free: https://docs.airnowapi.org/). Section 5 downloads ~100 MB the first time (cached afterwards).

Install: `pip install -e ".[dev,surface,models]" jupyter` from the repo root.

In [ ]:
import time, warnings
import pandas as pd
import atmoz
from atmoz.surface import AirNow, AirNowAPI, EPA_PreGen, read_aqs_zip
print("atmoz", atmoz.__version__)

## 1. Cache and credentials plumbing

In [ ]:
print(atmoz.cache_dir())          # where files go (override: ATMOZ_CACHE_DIR / ATMOZ_HOME)
print(atmoz.cache_info())         # {} on a fresh machine

### Save your AirNow key (run once; hidden prompt)
Writes `~/.atmoz/.atmoz.env` with mode 0600. Use `scope="project"` for `./.atmoz.env`.

In [ ]:
# AirNow.set_api_key()                      # prompts, hidden input
# AirNow.set_api_key(scope="project")
import os, stat
from atmoz.resources.credentials import find_env_file
f = find_env_file()
print("env file:", f, oct(stat.S_IMODE(os.stat(f).st_mode)) if f else None)   # expect 0o600

## 2. Unified `download()` return type (no key needed)
`AirNow.download()` must return an `atmoz_dataset` for **both** endpoints.

In [ ]:
t0 = time.time()
ds = AirNow.download(endpoint="aqs", parameters="ozone", resolutions="hourly", years=2024,
                     bbox=(-77.5, 38.5, -76.5, 39.5),
                     columns=["Latitude", "Longitude", "Date GMT", "Time GMT", "Sample Measurement"])
print(type(ds).__name__, ds.datatype, ds.metadata, f"{time.time()-t0:.1f}s (cold: downloads ~70 MB)")
df = ds.data["hourly_44201_2024.csv"]
print(df.shape); df.head()

In [ ]:
# Precision: lat/lon float64, negative signs kept, nothing rounded
print(df.dtypes)
print(df["Longitude"].min(), df["Longitude"].max())    # all negative

In [ ]:
t0 = time.time()
ds2 = AirNow.download(endpoint="aqs", parameters="ozone", resolutions="hourly", years=2024,
                      bbox=(-77.5, 38.5, -76.5, 39.5))
print(f"warm: {time.time()-t0:.1f}s")                   # expect ~1s (served from cache / revalidated)
atmoz.cache_info()

### All-or-nothing: a bad year must raise and return nothing

In [ ]:
try:
    AirNow.download(endpoint="aqs", parameters="ozone", resolutions="hourly", years=[2024, 1900], silent=True)
except atmoz.exceptions.DownloadError as e:
    print("OK, DownloadError:", str(e)[:200])

## 3. AirNow live API (needs a key)

### 3a. Does `endDate` include its hour?
`AirNow.window_end_inclusive = True` assumes it does. This asks for 00:00-02:00 and shows which hours come back.
If you see **00, 01, 02** the assumption is right. If you see **00, 01** set `AirNow.window_end_inclusive = False` and tell me.

In [ ]:
from atmoz.surface import AirNowAPI
payload = AirNowAPI()._fetch_one({"startDate": "2025-08-07T00", "endDate": "2025-08-07T02",
                                  "parameters": ["OZONE"], "BBOX": ["-77.5", "38.5", "-76.5", "39.5"]})
hours = sorted({row["UTC"] for row in payload})
print(hours)
print("inclusive" if any(h.endswith("02:00") for h in hours) else "exclusive")

### 3b. One day, several days, and `start == end`

In [ ]:
bbox = ["-77.5", "38.5", "-76.5", "39.5"]
one  = AirNow.download(start_date="2025-08-07", end_date="2025-08-07", parameters=["OZONE"], BBOX=bbox)
print("single day:", one.data.shape, one.data["UTC"].min(), "->", one.data["UTC"].max())   # expect 00:00 -> 23:00
week = AirNow.download(start_date="2025-08-01", end_date="2025-08-07", parameters=["OZONE", "PM25"], BBOX=bbox)
print("week:", week.data.shape, week.data["UTC"].nunique(), "distinct hours")                # expect 168
print("duplicated rows:", week.data.duplicated().sum())                                       # expect 0
week.metadata.head()

### 3c. Cache: second identical request should be near-instant

In [ ]:
t0 = time.time(); AirNow.download(start_date="2025-08-01", end_date="2025-08-07", parameters=["OZONE", "PM25"], BBOX=bbox)
print(f"{time.time()-t0:.2f}s")
atmoz.cache_info()

### 3d. Failure behaviour and the key never leaking
Wrong key -> a clean error, no partial data, and the key text must not appear in the message.

In [ ]:
bad = "NOT-A-REAL-KEY-123"
try:
    AirNow.download(api_key=bad, start_date="2025-08-01", end_date="2025-08-03", parameters=["OZONE"], BBOX=bbox, use_cache=False)
except Exception as e:
    print(type(e).__name__); print(str(e)[:400]); assert bad not in str(e) and bad not in repr(e)

## 4. Read an AQS zip you already have (no network)

In [ ]:
zips = sorted(atmoz.cache_dir("aqs").glob("hourly_44201_*.zip"))
print(zips)
if zips:
    d = read_aqs_zip(zips[0], columns=["Latitude", "Longitude", "Date GMT", "Sample Measurement"], bbox=(-77.5, 38.5, -76.5, 39.5))
    print(d.shape, d.dtypes.to_dict())

## 5. GEOS-CF (needs `atmoz[models]`; real OPeNDAP, can be slow)
Second call should load from `~/.atmoz/cache/geos_cf` in under a second.

In [ ]:
from atmoz.models import GEOS_CF
t0 = time.time()
g = GEOS_CF(); g.fetch(39.0, -76.9, "2024-06-01", "2024-06-03", variables=["o3"])
print(f"cold {time.time()-t0:.1f}s")
t0 = time.time()
g2 = GEOS_CF(); g2.fetch(39.0, -76.9, "2024-06-01", "2024-06-03", variables=["o3"])
print(f"warm {time.time()-t0:.1f}s")
print(atmoz.cache_info())

## 6. Clean up (optional)

In [ ]:
# atmoz.clear_cache("geos_cf")
# atmoz.clear_cache()      # everything